# Predicting next-month arrears
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), 5-split walk-forward CV on `billing_month` inside the training rows, selection by mean CV ROC AUC, holdout scored once.

**Feature decision.** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. after/alongside the outcome (arrears rate 87% vs 1.4%). It is not known at prediction time and is excluded as leakage. `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib, yaml
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

proto = yaml.safe_load(open('protocol.yaml'))
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest() == proto['dataset']['sha256']
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest() == proto['split']['holdout_sha256']
df = pd.read_csv('data.csv')
assert len(df) == proto['dataset']['rows']
TARGET = proto['target']
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(hold_idx) == proto['split']['holdout_rows']
train_idx = np.setdiff1d(np.arange(len(df)), hold_idx)
print(len(train_idx), len(hold_idx), df.loc[train_idx,'billing_month'].max(), df.loc[hold_idx,'billing_month'].min())

In [ ]:
NUM = ['occupants','kwh','bill_eur','late_payments_12m']
CAT = ['tariff']
FEATURES = NUM + CAT   # final_notice_sent excluded: recorded after the outcome window
train = df.loc[train_idx]
X, y = train[FEATURES], train[TARGET]
print(y.mean(), train.tariff.unique())

## Candidates (named before any scoring)

In [ ]:
def pre(scale):
    return ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', NUM),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
CANDIDATES = {
    'baseline_prior': Pipeline([('pre', pre(False)), ('m', DummyClassifier(strategy='prior'))]),
    'logreg_C1':      Pipeline([('pre', pre(True)),  ('m', LogisticRegression(C=1.0, max_iter=1000))]),
    'logreg_C0.1':    Pipeline([('pre', pre(True)),  ('m', LogisticRegression(C=0.1, max_iter=1000))]),
    'random_forest':  Pipeline([('pre', pre(False)), ('m', RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=0))]),
    'hist_gb':        Pipeline([('pre', pre(False)), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))]),
}
print(list(CANDIDATES))

## Walk-forward CV on billing month (training rows only)
Training months are split into 6 consecutive blocks; fold k fits on blocks 0..k and tests on block k+1. Preprocessing lives in the pipeline, so it is refit within each fold.

In [ ]:
months = np.sort(train['billing_month'].unique())
blocks = np.array_split(months, proto['cv']['n_splits'] + 1)
folds = []
for k in range(proto['cv']['n_splits']):
    tr_m = np.concatenate(blocks[:k+1]); te_m = blocks[k+1]
    tr = train_idx[train['billing_month'].isin(tr_m).to_numpy()]
    te = train_idx[train['billing_month'].isin(te_m).to_numpy()]
    folds.append((tr, te))
    print(k, len(tr), len(te), te_m[0], te_m[-1])

cv_scores = {}
for name, model in CANDIDATES.items():
    s = []
    for tr, te in folds:
        m = clone(model).fit(df.loc[tr, FEATURES], df.loc[tr, TARGET])
        s.append(float(roc_auc_score(df.loc[te, TARGET], m.predict_proba(df.loc[te, FEATURES])[:,1])))
    cv_scores[name] = s
cv_table = pd.DataFrame(cv_scores).T
cv_table['mean'] = cv_table.mean(axis=1)
cv_table.sort_values('mean', ascending=False)

## Selection and one-time holdout score

In [ ]:
chosen = max(cv_scores, key=lambda n: np.mean(cv_scores[n]))
print('chosen:', chosen, np.mean(cv_scores[chosen]))
final = clone(CANDIDATES[chosen]).fit(X, y)
hold = df.loc[hold_idx]
holdout_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))
print('HEADLINE holdout roc_auc:', holdout_auc)

In [ ]:
evidence = {
  'features': FEATURES,
  'candidates': list(CANDIDATES),
  'cv_folds': [[tr.tolist(), te.tolist()] for tr, te in folds],
  'cv_scores': cv_scores,
  'chosen': chosen,
  'holdout_rows': hold_idx.tolist(),
  'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
See the CV table and headline above. The headline holdout ROC AUC, using only features known at prediction time, is the honest estimate of predictive ability.